# Nightingale: EXP-006's weight sweep, then EXP-012's rehearsal, on Kaggle

The graph's weight α (EXP-006) has to be chosen on validation before the freeze, and the full ablation
(EXP-012) has to be rehearsed on validation with that α before the test split is opened once
(`docs/05` §2, `docs/08`). This notebook does both, on the CPU, in the background:
1. **EXP-006:** the sweep of α for every fused component (`scripts/sweep_fusion.py`). It writes
   `exp006_fusion.json`, whose chosen α goes into `configs/config.yaml`.
2. **EXP-012's rehearsal:** every system and ablation at 100%, 50% and 25% of the history, seeds 42-46,
   each fused component at its α (`scripts/evaluate_ablations.py`), with the verdicts under both readings
   of E-2. Its output is compared on the laptop with what EXP-019, EXP-021 and EXP-006 recorded: it must
   reproduce them exactly.

**Nothing is trained, and the test split is never downloaded.** The models come from a private dataset;
DDXPlus's validation split and BODHI-S's two files are fetched from Hugging Face and stay in `/tmp`
(BODHI-S is CC-BY-NC-4.0: research use only, never committed, never saved in the output).

## How to run it (about 10 minutes of your time)

1. **Once:** on kaggle.com, **Create → New Dataset**:
   - upload `models/kaggle_upload/nightingale_models_exp012.zip` from the laptop (53 MB);
   - name it `nightingale-models-exp012`;
   - keep it **Private**: models trained on DDXPlus are never published.
2. **Create → New Notebook → File → Import Notebook**: upload this file.
3. **Add Input → Your Datasets → nightingale-models-exp012.**
4. **Session options:**
   - **Accelerator: None.** The laptop's NumPy forward pass is what gets scored, so the CPU is enough.
   - **Internet: on.** It clones the repository and downloads the two datasets.
5. **Save Version → Save & Run All (Commit) → Save**, then close the browser.
   - It should take about 1 to 2 hours. This is an estimate: the sweep took about 8 minutes of laptop CPU
     in its pre-registration, the rehearsal is extrapolated from a 66-second run on 2,000 patients, and
     nothing like it has run on Kaggle's CPU yet.
6. When Kaggle emails you, open the latest version → **Output**:
   - download **`exp012_rehearsal.zip`**;
   - unzip it into `data/interim/` on the laptop, and tell Claude.

If a step fails, the version still saves `STATUS.txt` and the logs: download the zip anyway and send it.

In [ ]:
COMMIT = "master"
SEEDS = (42, 43, 44, 45, 46)
DRESS_REHEARSAL = False  # True only for a local check of this notebook's own commands: it shortens the long steps

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/tmp/nightingale")  # outside /kaggle/working, so no data ends up in the output
OUT = Path("/kaggle/working/exp012_rehearsal")
OUT.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True
).stdout.strip()
# The models were trained, and are scored, with the versions the laptop uses (their run.json files).
run(sys.executable, "-m", "pip", "install", "-q", "xgboost==2.1.4", "neo4j==5.28.6", "python-dotenv==1.2.3")

In [ ]:
# The models, from the private dataset added as input (found by folder name, zipped or not).
sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO))
from report_deep import place_models

BUNDLES = ("nightingale_b0_b1", "nightingale_b1_asked", "nightingale_b1_seeds", "nightingale_b1_dl")
found = place_models(Path("/kaggle/input"), REPO / "models", BUNDLES)
missing = [b for b in BUNDLES if not (REPO / "models" / b).exists()]
assert not missing, f"not found in /kaggle/input: {missing}. Add Input -> Your Datasets -> nightingale-models-exp012"
needed = [
    "nightingale_b0_b1/b0_prevalence.json",
    "nightingale_b0_b1/b1_logreg.json",
    "nightingale_b0_b1/b1_xgboost.ubj",
    "nightingale_b1_asked/b1_aug/b1_logreg.json",
    "nightingale_b1_asked/b1_asked_aug/b1_xgboost.ubj",
    "nightingale_b1_seeds/b1/seed46/b1_xgboost.ubj",
    "nightingale_b1_seeds/b1_aug/seed46/b1_logreg.json",
    "nightingale_b1_seeds/b1_asked_aug/seed46/b1_xgboost.ubj",
    "nightingale_b1_dl/b1_dl_aug/seed46/b1_dl.json",
    "nightingale_b1_dl/b1_dl_asked_aug/seed46/b1_dl.json",
]
absent = [n for n in needed if not (REPO / "models" / n).exists()]
assert not absent, f"the models dataset lacks {absent}"
print("placed:", found)

In [ ]:
# DDXPlus's validation split, at the snapshot the laptop uses (docs/11 §4).
# The test split is not fetched here: it opens once, after the freeze (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir=str(REPO / "data" / "raw" / "ddxplus"),
    )
# BODHI-S (CC-BY-NC-4.0, research use only): the two files the knowledge graph reads. They stay in /tmp.
bodhi = REPO / "data" / "raw" / "bodhi_s"
bodhi.mkdir(parents=True, exist_ok=True)
for name in ["triples.jsonl", "nl_facts.jsonl"]:
    got = hf_hub_download("ekacare/BODHI-S", f"data/{name}", repo_type="dataset", local_dir="/tmp/bodhi_download")
    shutil.copy(got, bodhi / name)
run(sys.executable, "scripts/decode_ddxplus.py", cwd=REPO)
run(sys.executable, "scripts/build_ddxplus_chestpain.py", "--split", "validate", cwd=REPO)

In [ ]:
# The graph must be the laptop's: the same sources give the same fingerprint (the one Aura holds).
from src.medical_kg.cardiac_kg import build_cardiac_kg
from src.medical_kg.neo4j_store import kg_fingerprint

interim = REPO / "data" / "interim"
kg = build_cardiac_kg(
    interim / "ddxplus_chestpain_conditions.json", interim / "ddxplus_evidences.json", bodhi
)
fingerprint = kg_fingerprint(kg)[:12]
print("graph fingerprint:", fingerprint)
assert fingerprint == "de349201157a", "this is not the graph the earlier experiments scored: stop here"

In [ ]:
# Smoke test: 300 patients, two seeds, an untuned alpha. A failure stops the version here. Its figures mean nothing.
run(sys.executable, "scripts/evaluate_ablations.py", "--limit", "300", "--untuned-alpha", "0.5",
    "--seeds", "42", "43", "--out", "/tmp/smoke.json", cwd=REPO)

In [ ]:
# EXP-006: the sweep. It records the golden-case gate first, then every alpha on the grid.
started = time.time()
sweep_args = ["--golden-only"] if DRESS_REHEARSAL else []
with open(OUT / "sweep.log", "w", encoding="utf-8") as handle:
    swept = subprocess.run(
        [sys.executable, "scripts/sweep_fusion.py", *sweep_args],
        cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
    )
sweep_minutes = (time.time() - started) / 60
sweep_status = "done" if swept.returncode == 0 else f"FAILED (exit {swept.returncode})"
sweep_file = REPO / "data" / "interim" / "exp006_fusion.json"
print(f"sweep {sweep_status} after {sweep_minutes:.0f} min")
print(open(OUT / "sweep.log", encoding="utf-8").read()[-2500:])

In [ ]:
# EXP-012's rehearsal: every system and ablation, every seed, at the alphas the sweep chose.
started = time.time()
rehearsal_status = "skipped: the sweep wrote no result"
if DRESS_REHEARSAL or sweep_file.exists():
    args = (
        ["--limit", "300", "--untuned-alpha", "0.5", "--seeds", "42", "43"]
        if DRESS_REHEARSAL
        else ["--alpha-file", str(sweep_file)]
    )
    with open(OUT / "rehearsal.log", "w", encoding="utf-8") as handle:
        rehearsed = subprocess.run(
            [sys.executable, "scripts/evaluate_ablations.py", *args,
             "--out", str(OUT / "exp012_validate_rehearsal.json")],
            cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
        )
    rehearsal_status = "done" if rehearsed.returncode == 0 else f"FAILED (exit {rehearsed.returncode})"
    print(open(OUT / "rehearsal.log", encoding="utf-8").read()[-3500:])
rehearsal_minutes = (time.time() - started) / 60
print(f"rehearsal {rehearsal_status} after {rehearsal_minutes:.0f} min")

In [ ]:
# What finished, and the one file to download (Output tab: exp012_rehearsal.zip).
import hashlib

import numpy
import pandas
import xgboost

if sweep_file.exists() and not DRESS_REHEARSAL:
    shutil.copy(sweep_file, OUT / "exp006_fusion.json")
facts = {
    "commit": commit,
    "graph_fingerprint": fingerprint,
    "sweep": f"{sweep_status}, {sweep_minutes:.0f} min",
    "rehearsal": f"{rehearsal_status}, {rehearsal_minutes:.0f} min",
    "dress_rehearsal": DRESS_REHEARSAL,
    "versions": {"python": sys.version.split()[0], "numpy": numpy.__version__,
                 "pandas": pandas.__version__, "xgboost": xgboost.__version__},
}
result = OUT / "exp012_validate_rehearsal.json"
if result.exists():
    data = json.loads(result.read_text(encoding="utf-8"))
    facts["rehearsal_valid_as_a_result"] = data.get("valid_as_a_result")
    facts["rehearsal_not_a_result_because"] = data.get("not_a_result_because")
    facts["digests"] = data.get("digests")
    facts["alphas"] = {k: v["alpha"] for k, v in data.get("alphas", {}).items()}
for name in ("exp006_fusion.json", "exp012_validate_rehearsal.json"):
    if (OUT / name).exists():
        facts.setdefault("sha256", {})[name] = hashlib.sha256((OUT / name).read_bytes()).hexdigest()[:16]
(OUT / "STATUS.txt").write_text(json.dumps(facts, indent=1, ensure_ascii=False), encoding="utf-8")
print(json.dumps(facts, indent=1, ensure_ascii=False))
archive = shutil.make_archive("/kaggle/working/exp012_rehearsal", "zip", OUT)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")